In [1]:
import pandas as pd 
import json

In [2]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)

# หาหน่วยเลือกตั้งของจังหวัด
rayong = next((p for p in data['provinces'] if p['name'] == 'ระยอง'), None)
# print(f"ระยอง {rayong['total_stations']} หน่วยเลือกตั้ง")
d = dict()
t = 0
# หาหน่วยเลือกตั้งจากรหัส
def find_total_station(area,province):
    for p in data['provinces']:
        if p['name'] == province:
            for a in p['areas']:
                if a['area'] == area:
                    for unit in a['stations']:
                        if unit['district'] not in d:
                            d[unit['district']] = dict()
                        if unit['subdistrict'] not in  d[unit['district']]:
                            d[unit['district']][unit['subdistrict']] = 1
                        else:
                            d[unit['district']][unit['subdistrict']] += 1
    return None

find_total_station(2, 'ลำปาง')
print(d)
for k, v in d.items():
    for e, g in v.items():
        t+=g
print(t)

{'เมืองลำปาง': {'บ้านแลง': 12, 'บ้านเสด็จ': 19}, 'งาว': {'หลวงเหนือ': 7, 'หลวงใต้': 8, 'บ้านโป่ง': 12, 'บ้านร้อง': 13, 'ปงเตา': 13, 'นาแก': 6, 'บ้านอ้อน': 8, 'บ้านแหง': 9, 'บ้านหวด': 7, 'แม่ตีบ': 7}, 'แจ้ห่ม': {'แจ้ห่ม': 14, 'บ้านสา': 10, 'ปงดอน': 9, 'แม่สุก': 12, 'เมืองมาย': 7, 'ทุ่งผึ้ง': 8, 'วิเชตนคร': 12}, 'วังเหนือ': {'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}, 'เมืองปาน': {'เมืองปาน': 11, 'บ้านขอ': 14, 'ทุ่งกว๋าว': 16, 'แจ้ซ้อน': 15, 'หัวเมือง': 9}}
342


In [3]:
all_distrcit = []
all_subdistrict = []
district_count = {}
for k,v in d.items():
    all_distrcit.append(k)
    all_subdistrict += v.keys()
    for key in v.keys():
        if k not in district_count:
            district_count[k] = v[key]
        else: district_count[k]+= v[key]

In [4]:
district_count

{'เมืองลำปาง': 31, 'งาว': 90, 'แจ้ห่ม': 72, 'วังเหนือ': 84, 'เมืองปาน': 65}

In [5]:
summary = pd.read_csv("../master_summary.csv")
summary.head(5)

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,663,8,29,-1
1,เขต,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,605,32,63,-1
2,บช,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,668,8,24,-1
3,เขต,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,616,28,56,-1
4,บช,ลำปาง,นอกเขต,ชุดที่ 11,-1,700,664,6,30,-1


In [6]:
results = pd.read_csv("../master_results.csv")
results.head(5)

,type,province,district,sub-district,unit_number,name,score
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ไทยทรัพย์ทวี,0
1,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,เพื่อชาติไทย,6
2,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,ใหม่,4
3,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,มิติใหม่,0
4,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,รวมใจไทย,1


In [7]:
summary.info()

<class 'pandas.DataFrame'>
RangeIndex: 710 entries, 0 to 709
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   type             710 non-null    str  
 1   province         710 non-null    str  
 2   district         710 non-null    str  
 3   sub-district     710 non-null    str  
 4   unit_number      710 non-null    int64
 5   total_ballots    710 non-null    int64
 6   valid_ballots    710 non-null    int64
 7   invalid_ballots  710 non-null    int64
 8   no_vote_ballots  710 non-null    int64
 9   remain_ballots   710 non-null    int64
dtypes: int64(6), str(4)
memory usage: 55.6 KB


In [8]:
for col in summary.columns:
    if summary[(summary[col] == '-1') | (summary[col] == -1)][col].sum():
        print(f"with column: {col} have -1 total be: {summary[(summary[col] == '-1') | (summary[col] == -1)][col].sum()}")

with column: unit_number have -1 total be: -26
with column: total_ballots have -1 total be: -2
with column: invalid_ballots have -1 total be: -7
with column: no_vote_ballots have -1 total be: -11
with column: remain_ballots have -1 total be: -164


In [9]:
fix_district={
}
for district in all_distrcit:
    df_district = summary[summary['district'].str.contains(district)]['district']
    if len(df_district)/2 != district_count[district]:
        print(f"District: {district} have total unit: {len(df_district)} but target is: {district_count[district]}")
    for miss_district in df_district.unique():
        if miss_district!= district:
            fix_district[miss_district] = district

District: งาว have total unit: 176 but target is: 90


In [10]:
for miss_district, real in fix_district.items():
    summary.loc[summary['district'] == miss_district, 'district'] = real
for district in all_distrcit:
    print(summary[summary['district'].str.contains(district)]['district'].unique())

<StringArray>
['เมืองลำปาง']
Length: 1, dtype: str
<StringArray>
['งาว']
Length: 1, dtype: str
<StringArray>
['แจ้ห่ม']
Length: 1, dtype: str
<StringArray>
['วังเหนือ']
Length: 1, dtype: str
<StringArray>
['เมืองปาน']
Length: 1, dtype: str


In [11]:
# fix sub-district
counts = summary['sub-district'].value_counts()

result = counts[counts == 1].index.tolist()
summary[summary['sub-district'].isin(result)]


,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
418,เขต,ลำปาง,เมืองปาน,เจริญยุทธ,10,120,66,5,1,43
422,เขต,ลำปาง,เมืองปาน,เจ้าซ้อน,14,320,217,11,1,91
427,เขต,ลำปาง,เมืองปาน,แจ้งขึ้น,5,460,313,24,12,111
430,เขต,ลำปาง,เมืองปาน,เจ็ดสิบ,8,600,375,24,11,-1
439,บช,ลำปาง,เมืองปาน,ทุ่งกว้าง,16,800,180,25,11,-1


In [12]:
c = summary[summary['district'] == 'เมืองปาน']['sub-district'].value_counts().index.tolist()
c

['ทุ่งกว๋าว',
 'บ้านขอ',
 'เมืองปาน',
 'หัวเมือง',
 'แจ้ซ้อน',
 'เจ้ซ้อน',
 'เจริญยุทธ',
 'เจ้าซ้อน',
 'แจ้งขึ้น',
 'เจ็ดสิบ',
 'ทุ่งกว้าง']

In [13]:
c[5:-1]

['เจ้ซ้อน', 'เจริญยุทธ', 'เจ้าซ้อน', 'แจ้งขึ้น', 'เจ็ดสิบ']

In [14]:
summary.loc[summary['sub-district'].isin(c[5:-1]), 'sub-district'] = c[4]
summary.loc[summary['sub-district'].isin([c[-1]]), 'sub-district'] = c[0]

In [15]:
summary[summary['district'] == 'เมืองปาน']['sub-district'].value_counts()

sub-district
ทุ่งกว๋าว    32
แจ้ซ้อน      30
บ้านขอ       28
เมืองปาน     22
หัวเมือง     18
Name: count, dtype: int64

In [16]:
summary.loc[summary['province'] != "ลำปาง", "province"] = "ลำปาง"
summary['province'].unique()

<StringArray>
['ลำปาง']
Length: 1, dtype: str

In [17]:
anomaly_subdistrict = []
for district in all_distrcit:
    l = summary[summary['district'] == district]['sub-district'].value_counts().index.tolist()
    l_check = list(d[district].keys())
    for it in l:
        if it not in l_check:
            print(l_check)
            print(f"District: {district} have anomaly sub-district: {it}")
            anomaly_subdistrict.append(it)
    # if l != l_check:
    #     print(district)

['ทุ่งฮั้ว', 'วังเหนือ', 'วังใต้', 'ร่องเคาะ', 'วังทอง', 'วังซ้าย', 'วังแก้ว', 'วังทรายคำ']
District: วังเหนือ have anomaly sub-district: บ้านใหม่


In [18]:
summary[summary['sub-district'].isin(anomaly_subdistrict)]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
218,เขต,ลำปาง,วังเหนือ,บ้านใหม่,1,460,297,34,12,117
219,เขต,ลำปาง,วังเหนือ,บ้านใหม่,2,420,293,21,14,-1
220,เขต,ลำปาง,วังเหนือ,บ้านใหม่,3,540,333,16,21,160
221,เขต,ลำปาง,วังเหนือ,บ้านใหม่,4,560,400,70,17,160
295,บช,ลำปาง,วังเหนือ,บ้านใหม่,1,540,350,17,13,160
296,บช,ลำปาง,วังเหนือ,บ้านใหม่,2,420,328,20,9,-1
297,บช,ลำปาง,วังเหนือ,บ้านใหม่,3,460,315,21,7,-1
298,บช,ลำปาง,วังเหนือ,บ้านใหม่,4,560,378,32,12,160


In [19]:
wang_north = summary[summary['district'] == 'วังเหนือ']['sub-district'].unique()
print(wang_north)
print(d['วังเหนือ'])
summary[(summary['district'] == 'วังเหนือ') & (summary['sub-district'] == 'วังเหนือ')]
# for sub_dis, val in d['วังเหนือ'].items():
#     if 

<StringArray>
[   'วังใต้',  'ทุ่งฮั้ว',  'บ้านใหม่',  'ร่องเคาะ',   'วังซ้าย', 'วังทรายคำ',
    'วังทอง',  'วังเหนือ',   'วังแก้ว']
Length: 9, dtype: str
{'ทุ่งฮั้ว': 12, 'วังเหนือ': 12, 'วังใต้': 7, 'ร่องเคาะ': 18, 'วังทอง': 9, 'วังซ้าย': 10, 'วังแก้ว': 7, 'วังทรายคำ': 9}


,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
268,เขต,ลำปาง,วังเหนือ,วังเหนือ,1,200,134,8,50,50
269,เขต,ลำปาง,วังเหนือ,วังเหนือ,2,240,163,13,9,55
270,เขต,ลำปาง,วังเหนือ,วังเหนือ,3,780,463,24,52,461
271,เขต,ลำปาง,วังเหนือ,วังเหนือ,4,600,429,41,42,771
272,เขต,ลำปาง,วังเหนือ,วังเหนือ,5,360,216,17,29,98
273,เขต,ลำปาง,วังเหนือ,วังเหนือ,6,420,280,34,7,99
274,เขต,ลำปาง,วังเหนือ,วังเหนือ,7,100,125,15,18,44
275,เขต,ลำปาง,วังเหนือ,วังเหนือ,8,440,298,18,27,97
345,บช,ลำปาง,วังเหนือ,วังเหนือ,1,200,137,13,-1,50
346,บช,ลำปาง,วังเหนือ,วังเหนือ,2,240,185,23,7,55


In [20]:
with open('../election-stations-2569.json', 'r', encoding='utf-8') as f:
    data = json.load(f)
s = set()
for p in data['provinces']:
    if p['name'] == "ลำปาง":
        for a in p['areas']:
            if a['area'] == 2:
                for unit in a['stations']:
                    if unit['district'] == 'วังเหนือ':
                        print(unit)                    

{'code': '52-02-520701-001', 'name': 'อาคารศาลาประชาคมหมู่บ้านบ้านทุ่งปี้ หมู่ที่ 1', 'district': 'วังเหนือ', 'subdistrict': 'ทุ่งฮั้ว'}
{'code': '52-02-520701-002', 'name': 'อาคารอเนกประสงค์วัดแม่ทรายเงิน หมู่ที่ 2', 'district': 'วังเหนือ', 'subdistrict': 'ทุ่งฮั้ว'}
{'code': '52-02-520701-003', 'name': 'อาคารอเนกประสงค์บ้านผาดิน หมู่ที่ 3', 'district': 'วังเหนือ', 'subdistrict': 'ทุ่งฮั้ว'}
{'code': '52-02-520701-004', 'name': 'อาคารอเนกประสงค์บ้านทุ่งฮั้ว หมู่ที่ 4', 'district': 'วังเหนือ', 'subdistrict': 'ทุ่งฮั้ว'}
{'code': '52-02-520701-005', 'name': 'โรงเรียนบ้านวังมน หมู่ที่ 5', 'district': 'วังเหนือ', 'subdistrict': 'ทุ่งฮั้ว'}
{'code': '52-02-520701-006', 'name': 'อาคารอเนกประสงค์บ้านผาแดง หมู่ที่ 6', 'district': 'วังเหนือ', 'subdistrict': 'ทุ่งฮั้ว'}
{'code': '52-02-520701-007', 'name': 'ศาลาการเปรียญวัดห้วยกันทา หมู่ที่ 7', 'district': 'วังเหนือ', 'subdistrict': 'ทุ่งฮั้ว'}
{'code': '52-02-520701-008', 'name': 'อาคารอเนกประสงค์บ้านแม่เลียบ หมู่ที่ 8', 'district': 'วังเหนือ'

In [21]:
# แปลกตรงที่ jsn election station ไม่มีตำบลบ้ายใหม่ แต่ใน pdf มี บ้านใหม่แล้วก็ วังเหนือมีแต่ 8 แต่น json มี 12 หน่วย
check = summary[summary['district'] == 'งาว'].groupby('sub-district')['unit_number'].count()
miss_subdistrict = []
for key, val in d['งาว'].items():
    if check[key]/2 != val:
        print(f"ตำบล: {key}, target: {val}, but have only: {check[key]/2}")
        miss_subdistrict.append(key)

ตำบล: หลวงใต้, target: 8, but have only: 7.5
ตำบล: นาแก, target: 6, but have only: 5.0
ตำบล: บ้านอ้อน, target: 8, but have only: 7.5


In [22]:
miss_subdistrict[0]

'หลวงใต้'

In [23]:
summary[(summary['district'] == 'งาว') & (summary['sub-district'] == miss_subdistrict[0])]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
94,เขต,ลำปาง,งาว,หลวงใต้,1,600,306,25,13,194
95,เขต,ลำปาง,งาว,หลวงใต้,2,740,405,32,25,278
96,เขต,ลำปาง,งาว,หลวงใต้,3,480,323,14,14,129
97,เขต,ลำปาง,งาว,หลวงใต้,4,360,259,72,17,-1
98,เขต,ลำปาง,งาว,หลวงใต้,5,580,354,20,14,192
99,เขต,ลำปาง,งาว,หลวงใต้,6,760,443,15,13,299
100,เขต,ลำปาง,งาว,หลวงใต้,7,340,223,12,4,101
101,เขต,ลำปาง,งาว,หลวงใต้,8,420,276,22,7,115
159,บช,ลำปาง,งาว,หลวงใต้,1,600,373,26,7,-1
160,บช,ลำปาง,งาว,หลวงใต้,2,740,462,28,15,278


In [24]:
summary[(summary['district'] == 'งาว') & (summary['sub-district'] == miss_subdistrict[1])]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
24,เขต,ลำปาง,งาว,นาแก,1,300,204,8,2,86
25,เขต,ลำปาง,งาว,นาแก,2,700,404,27,17,252
26,เขต,ลำปาง,งาว,นาแก,3,360,219,14,11,116
28,เขต,ลำปาง,งาว,นาแก,5,640,437,29,15,159
29,เขต,ลำปาง,งาว,นาแก,6,230,177,15,0,88
109,บช,ลำปาง,งาว,นาแก,1,300,198,8,0,94
110,บช,ลำปาง,งาว,นาแก,2,700,448,22,13,252
111,บช,ลำปาง,งาว,นาแก,3,360,249,111,16,116
113,บช,ลำปาง,งาว,นาแก,5,640,449,21,11,159
114,บช,ลำปาง,งาว,นาแก,6,280,183,9,0,88


In [25]:
summary[(summary['district'] == 'งาว') & (summary['sub-district'] == miss_subdistrict[2])]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
49,เขต,ลำปาง,งาว,บ้านอ้อน,1,301,269,18,14,-1
50,เขต,ลำปาง,งาว,บ้านอ้อน,3,360,368,12,11,-1
51,เขต,ลำปาง,งาว,บ้านอ้อน,4,500,322,19,6,153
52,เขต,ลำปาง,งาว,บ้านอ้อน,5,400,158,27,11,204
53,เขต,ลำปาง,งาว,บ้านอ้อน,6,420,291,18,6,125
54,เขต,ลำปาง,งาว,บ้านอ้อน,8,160,104,8,0,48
135,บช,ลำปาง,งาว,บ้านอ้อน,1,440,391,7,9,-1
136,บช,ลำปาง,งาว,บ้านอ้อน,2,1000,385,31,7,-1
137,บช,ลำปาง,งาว,บ้านอ้อน,3,560,391,12,9,169
138,บช,ลำปาง,งาว,บ้านอ้อน,4,500,325,5,153,-1


In [26]:
summary[~summary['district'].isin(all_distrcit)]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,663,8,29,-1
1,เขต,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,605,32,63,-1
2,บช,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,668,8,24,-1
3,เขต,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,616,28,56,-1
4,บช,ลำปาง,นอกเขต,ชุดที่ 11,-1,700,664,6,30,-1
5,เขต,ลำปาง,นอกเขต,ชุดที่ 11,-1,700,420,28,65,-1
6,บช,ลำปาง,นอกเขต,ชุดที่ 12,-1,700,666,7,29,-1
7,เขต,ลำปาง,นอกเขต,ชุดที่ 12,-1,700,614,33,53,-1
8,บช,ลำปาง,นอกเขต,ชุดที่ 13,-1,694,662,0,23,-1
9,เขต,ลำปาง,นอกเขต,ชุดที่ 13,-1,694,599,37,58,-1


In [27]:
summary.loc[[27, 112, 699, 161], 'district'] = 'งาว' 

In [28]:
summary[~summary['district'].isin(all_distrcit)]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
0,บช,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,663,8,29,-1
1,เขต,ลำปาง,นอกเขต,ชุดที่ 1,-1,700,605,32,63,-1
2,บช,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,668,8,24,-1
3,เขต,ลำปาง,นอกเขต,ชุดที่ 10,-1,700,616,28,56,-1
4,บช,ลำปาง,นอกเขต,ชุดที่ 11,-1,700,664,6,30,-1
5,เขต,ลำปาง,นอกเขต,ชุดที่ 11,-1,700,420,28,65,-1
6,บช,ลำปาง,นอกเขต,ชุดที่ 12,-1,700,666,7,29,-1
7,เขต,ลำปาง,นอกเขต,ชุดที่ 12,-1,700,614,33,53,-1
8,บช,ลำปาง,นอกเขต,ชุดที่ 13,-1,694,662,0,23,-1
9,เขต,ลำปาง,นอกเขต,ชุดที่ 13,-1,694,599,37,58,-1


In [31]:
for dist in all_distrcit:
    check = summary[summary['district'] == dist].groupby('sub-district')['unit_number'].count()
    miss_subdistrict = []
    for key, val in d[dist].items():
        if check[key]/2 != val:
            print(f"ตำบล: {key}, target: {val}, but have only: {check[key]/2}")
            miss_subdistrict.append(key)
    print(miss_subdistrict)

[]
[]
[]
ตำบล: วังเหนือ, target: 12, but have only: 8.0
['วังเหนือ']
[]


In [32]:
# วังเหนือใน pdf มีแ่ 8 แต่บ้านใหม่หาย ซึ่ง pdf มีบ้านใหม่ 4
for col in summary.columns:
    if summary[(summary[col] == '-1') | (summary[col] == -1)][col].sum():
        print(f"with column: {col} have -1 total be: {summary[(summary[col] == '-1') | (summary[col] == -1)][col].sum()}")


with column: unit_number have -1 total be: -26
with column: total_ballots have -1 total be: -2
with column: invalid_ballots have -1 total be: -7
with column: no_vote_ballots have -1 total be: -11
with column: remain_ballots have -1 total be: -164


# Mannual Validate

In [33]:
summary[summary['total_ballots'] == -1]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
175,บช,ลำปาง,งาว,บ้านโป่ง,10,-1,279,-1,-1,-1
263,เขต,ลำปาง,วังเหนือ,วังทอง,5,-1,117,12,10,43


In [34]:
summary.loc[[175], ["total_ballots", "invalid_ballots", "no_vote_ballots", "remain_ballots"]] = [460, 14, 9, 158]
summary.loc[175]

type                     บช
province              ลำปาง
district                งาว
sub-district       บ้านโป่ง
unit_number              10
total_ballots           460
valid_ballots           279
invalid_ballots          14
no_vote_ballots           9
remain_ballots          158
Name: 175, dtype: object

In [36]:
cols = summary.columns.tolist()
cols

['type',
 'province',
 'district',
 'sub-district',
 'unit_number',
 'total_ballots',
 'valid_ballots',
 'invalid_ballots',
 'no_vote_ballots',
 'remain_ballots']

In [37]:
summary.loc[[263], cols[5:]] = [180, 115, 12, 10, 43]
summary.loc[263]

type                    เขต
province              ลำปาง
district           วังเหนือ
sub-district         วังทอง
unit_number               5
total_ballots           180
valid_ballots           115
invalid_ballots          12
no_vote_ballots          10
remain_ballots           43
Name: 263, dtype: object

In [39]:
summary[summary[cols[7]] == -1]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
55,เขต,ลำปาง,งาว,บ้านแหง,1,60,395,-1,30,-1
153,บช,ลำปาง,งาว,หลวงเหนือ,2,240,279,-1,-1,-1
501,เขต,ลำปาง,เมืองลำปาง,บ้านเสด็จ,15,480,279,-1,-1,133
625,เขต,ลำปาง,แจ้ห่ม,ปงดอน,8,300,159,-1,109,21
686,บช,ลำปาง,แจ้ห่ม,แม่สุก,10,200,377,-1,-1,-1
689,บช,ลำปาง,แจ้ห่ม,แม่สุก,2,620,462,-1,-1,-1


In [40]:
summary.loc[[55], cols[5:]] = [600, 395, 26, 19, 220]
summary.loc[55]

type                   เขต
province             ลำปาง
district               งาว
sub-district       บ้านแหง
unit_number              1
total_ballots          600
valid_ballots          395
invalid_ballots         26
no_vote_ballots         19
remain_ballots         220
Name: 55, dtype: object

In [41]:
summary.loc[[153], cols[5:]] = [440, 274, 12, 22, 132]
summary.loc[153]

type                      บช
province               ลำปาง
district                 งาว
sub-district       หลวงเหนือ
unit_number                2
total_ballots            440
valid_ballots            274
invalid_ballots           12
no_vote_ballots           22
remain_ballots           132
Name: 153, dtype: object

In [47]:
check_cols = cols[5:]
summary[check_cols] = summary[check_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

mask = summary[cols[6:]].sum(axis=1) != summary[cols[5]]
error_rows = summary[mask]

In [56]:
error_index = error_rows.iloc[24:].index.tolist()

In [60]:
all_subdistrict.append("บ้านใหม่")

In [61]:
summary[summary['sub-district'] == all_subdistrict[0]]

,type,province,district,sub-district,unit_number,total_ballots,valid_ballots,invalid_ballots,no_vote_ballots,remain_ballots
514,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,1,420,240,33,11,-1
515,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,10,240,144,12,10,74
516,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,11,706,479,50,23,-1
517,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,12,270,159,31,10,80
518,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,2,320,183,12,20,105
519,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,3,700,335,58,47,158
520,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,4,500,319,87,22,134
521,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,5,420,292,39,14,75
522,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,6,280,172,18,9,81
523,เขต,ลำปาง,เมืองลำปาง,บ้านแลง,7,240,122,29,5,28


In [ ]:
summary.to_csv("master_summary_1.csv")

In [63]:
for sub_dsit in all_subdistrict:
    print(sub_dsit)

บ้านแลง
บ้านเสด็จ
หลวงเหนือ
หลวงใต้
บ้านโป่ง
บ้านร้อง
ปงเตา
นาแก
บ้านอ้อน
บ้านแหง
บ้านหวด
แม่ตีบ
แจ้ห่ม
บ้านสา
ปงดอน
แม่สุก
เมืองมาย
ทุ่งผึ้ง
วิเชตนคร
ทุ่งฮั้ว
วังเหนือ
วังใต้
ร่องเคาะ
วังทอง
วังซ้าย
วังแก้ว
วังทรายคำ
เมืองปาน
บ้านขอ
ทุ่งกว๋าว
แจ้ซ้อน
หัวเมือง
บ้านใหม่
